# Genius seed → YouTube → Demucs → MOHIM → ACE-Step LoRA

이 노트북은 Colab 전용입니다. 로컬에서 만든 `genius_pop_seed.json`을 Google Drive에 올린 뒤, YouTube 음원 다운로드부터 Demucs 분리, 모티프 데이터셋 생성, LoRA 학습까지 순서대로 실행합니다.

## 0. Drive 마운트와 저장소 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'working'
REPO_DIR = Path('/content/MOHIM')

if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q --no-deps "beat-this @ git+https://github.com/CPJKU/beat_this.git"

# swift-f0는 PyPI에 존재하는 0.1.2를 사용하고 beat-this만 --no-deps로 설치합니다.
# 첫 설치 후 import 오류가 나면 런타임을 한 번 재시작하고 이 셀부터 다시 실행하세요.

## 1. 경로와 실행 범위 설정

In [ ]:
from pathlib import Path

BASE_LORA_VERSION = 'v3'    # rank 64 + both attention + validation 학습 버전
SMOKE_TEST = True           # 3곡/1 epoch end-to-end 검증 후 본학습 때 False
LORA_VERSION = f'{BASE_LORA_VERSION}_smoke' if SMOKE_TEST else BASE_LORA_VERSION
LORA_RUN_ROOT = Path('/content/drive/MyDrive/MOHIM/lora_run')
LORA_OUTPUT_DIR = LORA_RUN_ROOT / LORA_VERSION
DRIVE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/MOHIM/mohim_lora_checkpoints')
DRIVE_CHECKPOINT_DIR = DRIVE_CHECKPOINT_ROOT / LORA_VERSION
INFERENCE_OUTPUT_DIR = Path('/content/drive/MyDrive/MOHIM/inference') / LORA_VERSION

DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/genius_pop_dataset')
SEED_JSON = DATASET_DIR / 'genius_pop_seed.json'
TRACKS_JSON = DATASET_DIR / 'tracks.json'
AUDIO_DIR = DATASET_DIR / 'audio'
OUTPUT_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset')
MANIFEST_PATH = OUTPUT_DIR / (
    'dual_stream_manifest_smoke.json' if SMOKE_TEST else 'dual_stream_manifest.json'
)
BEAT_CHECKPOINT = Path('/content/checkpoints/beat_this_final0.ckpt')

SMOKE_SONGS = 3
MAX_DOWNLOADS = SMOKE_SONGS if SMOKE_TEST else None
YOUTUBE_SEARCH_RESULTS = 8  # unavailable이면 다음 후보를 시도
MAX_TRACK_DURATION = 300.0  # 5분 초과 곡은 tracks.json에 넣지 않음
MAX_SONGS = SMOKE_SONGS if SMOKE_TEST else None
RESET_TENSORS = False       # 기존 240초 tensor를 지우는 최초 1회만 True
RESET_LORA_RUN = False      # 기존 smoke checkpoint를 지우는 최초 1회만 True
DEVICE = 'cuda'
AUDIO_FORMAT = 'flac'
MOTIF_BARS = 4
MOTIF_SIMILARITY = 0.56
MOTIF_SEARCH_SECONDS = 45.0

assert SEED_JSON.is_file(), f'Drive에 seed JSON을 올려주세요: {SEED_JSON}'
DATASET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('seed:', SEED_JSON)
print('audio:', AUDIO_DIR)
print('processed:', OUTPUT_DIR)

## 2. seed JSON과 YouTube 음원 연결

제목과 아티스트로 여러 후보를 검색하고 커버·라이브·리믹스를 제외합니다. unavailable 후보는 건너뛰며, 성공한 곡마다 `tracks.json`을 즉시 저장합니다. 재실행하면 이미 받은 음원은 건너뜁니다.

In [ ]:
import importlib
import mohim.local_dataset as local_dataset
local_dataset = importlib.reload(local_dataset)

download_manifest = local_dataset.ingest_genius_seed(
    SEED_JSON,
    DATASET_DIR,
    max_tracks=MAX_DOWNLOADS,
    search_results=YOUTUBE_SEARCH_RESULTS,
    max_duration_seconds=MAX_TRACK_DURATION,
)
tracks = local_dataset.load_local_tracks(
    TRACKS_JSON,
    require_lyrics=True,
    max_duration_seconds=MAX_TRACK_DURATION,
)
print('downloaded tracks:', len(tracks))
print('current-run errors:', len(download_manifest['errors']))
assert tracks, '다운로드된 트랙이 없습니다. 위 오류와 tracks.json을 확인하세요.'

In [ ]:
import pandas as pd

preview = pd.DataFrame([
    {
        'track_id': track.track_id,
        'artist': track.artist,
        'title': track.title,
        'duration_seconds': track.duration_seconds,
        'audio_path': track.audio_path,
        'lyrics_chars': len(track.lyrics),
    }
    for track in tracks[:20]
])
display(preview)

## 3. 다운로드 결과 한 곡 확인

In [ ]:
from IPython.display import Audio, display

audio_index = local_dataset.index_audio_files(AUDIO_DIR)
matched = [(track, local_dataset.resolve_audio_path(track, audio_index)) for track in tracks]
matched = [(track, path) for track, path in matched if path is not None]
assert matched, 'tracks.json의 audio_path와 Drive의 audio 폴더를 확인하세요.'
sample_track, sample_audio_path = matched[0]
print(sample_track.track_id, sample_track.artist, '-', sample_track.title)
display(Audio(filename=str(sample_audio_path)))

## 4. HTDemucs 6-stem 분리 확인

In [ ]:
from mohim.separator import StemSeparator

separator = StemSeparator(device=DEVICE, model_name='htdemucs_6s')
stems, sample_rate, mixture = separator.separate(sample_audio_path)
print('sample rate:', sample_rate)
print('stems:', list(stems))

In [ ]:
for stem_name in ('vocals', 'guitar', 'piano', 'bass', 'other', 'drums'):
    if stem_name in stems:
        print('---', stem_name, '---')
        display(Audio(stems[stem_name].numpy(), rate=sample_rate))

## 5. Beat This와 반복 4마디 모티프 확인

In [ ]:
import urllib.request

BEAT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not BEAT_CHECKPOINT.is_file():
    urllib.request.urlretrieve(
        'https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt',
        BEAT_CHECKPOINT,
    )
print(BEAT_CHECKPOINT, BEAT_CHECKPOINT.stat().st_size, 'bytes')

In [ ]:
from mohim.motif import MotifConfig, MotifExtractor, create_beat_tracker

beat_tracker = create_beat_tracker(BEAT_CHECKPOINT, device=DEVICE)
motif_extractor = MotifExtractor(
    beat_tracker,
    MotifConfig(
        bars=MOTIF_BARS,
        search_seconds=MOTIF_SEARCH_SECONDS,
        similarity_threshold=MOTIF_SIMILARITY,
    ),
)
motif_result = motif_extractor.extract(sample_audio_path, stems, mixture, sample_rate)
print('selected stem:', motif_result['stem_name'])
print('segment:', motif_result['start_sec'], '~', motif_result['end_sec'])
print('repeat similarity:', motif_result['similarity'])
display(pd.DataFrame(motif_result['stem_scores']).T.sort_values('total', ascending=False))
display(Audio(motif_result['audio'].numpy(), rate=sample_rate))

## 6. Demucs 분리와 모티프 데이터셋 배치 생성

`MAX_SONGS = 3`으로 결과를 먼저 확인한 뒤 `None`으로 바꾸세요. 샘플별 `metadata.json`이 완성된 곡은 재실행 시 건너뜁니다.

In [ ]:
from dataclasses import asdict
from mohim.dataset import DatasetBuilder

builder = DatasetBuilder(
    audio_dir=AUDIO_DIR,
    output_dir=OUTPUT_DIR,
    separator=separator,
    motif_extractor=motif_extractor,
    audio_format=AUDIO_FORMAT,
    resume=True,
)
results = builder.build([track for track, _ in matched], max_songs=MAX_SONGS)
results_df = pd.DataFrame([asdict(result) for result in results])
display(results_df)
display(results_df.groupby(['status', 'reason'], dropna=False).size().rename('count').reset_index())

## 7. ACE-Step dual-stream manifest 생성

In [ ]:
import json
from mohim.manifest import build_dual_stream_manifest

manifest_track_ids = (
    {result.track_id for result in results}
    if SMOKE_TEST
    else {track.track_id for track in tracks}
)
manifest = build_dual_stream_manifest(
    OUTPUT_DIR,
    MANIFEST_PATH,
    allowed_track_ids=manifest_track_ids,
)
if SMOKE_TEST:
    assert manifest['metadata']['num_samples'] == SMOKE_SONGS, (
        f'smoke manifest는 {SMOKE_SONGS}곡이어야 합니다: '
        f"{manifest['metadata']['num_samples']}곡"
    )
print('usable samples:', manifest['metadata']['num_samples'])
print('manifest:', MANIFEST_PATH)
if manifest['samples']:
    print(json.dumps(manifest['samples'][0], ensure_ascii=False, indent=2)[:3000])

## 8. 공식 ACE-Step clone 및 dual-stream 패치 적용

In [ ]:
from mohim.trainer import (
    DEFAULT_REVISION,
    apply_acestep_patch,
    ensure_acestep_repo,
    install_acestep,
)

ACESTEP_DIR = Path('/content/ACE-Step-1.5')
PATCH_FILE = REPO_DIR / 'patches/ace-step-1.5-dual-stream.patch'
ACESTEP_DIR = ensure_acestep_repo(ACESTEP_DIR, revision=DEFAULT_REVISION)
apply_acestep_patch(ACESTEP_DIR, PATCH_FILE)
INSTALL_ACESTEP = True
if INSTALL_ACESTEP:
    install_acestep(ACESTEP_DIR)
# Colab에서 검증된 PyTorch 2.10 조합과 기본 패키지 버전을 복구합니다.
%pip uninstall -q -y huggingface-hub
%pip install -q --force-reinstall --no-cache-dir --no-deps \
    huggingface-hub==0.36.0 requests==2.32.4 fsspec==2025.3.0 \
    torchcodec==0.10.0 torchao==0.16.0

# 재설치 전 실패한 Hugging Face import가 현재 커널에 남아 있으면 제거합니다.
import importlib
import sys
for module_name in list(sys.modules):
    if module_name == 'huggingface_hub' or module_name.startswith('huggingface_hub.'):
        del sys.modules[module_name]
    elif module_name == 'transformers' or module_name.startswith('transformers.'):
        del sys.modules[module_name]
    elif module_name == 'peft' or module_name.startswith('peft.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
print('ACE-Step directory:', ACESTEP_DIR)

## 9. Dual-stream tensor 전처리

In [ ]:
import shutil

CHECKPOINT_DIR = Path('/content/drive/MyDrive/MOHIM/checkpoints')
TENSOR_DIR = Path('/content/drive/MyDrive/MOHIM') / (
    'dual_stream_tensors_smoke' if SMOKE_TEST else 'dual_stream_tensors'
)
TENSOR_SCHEMA = 'branch_prompts_v2'
TENSOR_SCHEMA_PATH = TENSOR_DIR / '.mohim_schema'
PREPROCESS_RUN_DIR = ACESTEP_DIR / 'mohim_preprocess_run'
MODEL_VARIANT = 'base'
MAX_DURATION = MAX_TRACK_DURATION

# 공통 prompt tensor는 branch별 prompt tensor와 호환되지 않아 최초 한 번 자동 교체됩니다.
tensor_schema_changed = (
    TENSOR_DIR.exists()
    and (not TENSOR_SCHEMA_PATH.is_file() or TENSOR_SCHEMA_PATH.read_text().strip() != TENSOR_SCHEMA)
)
if (SMOKE_TEST or RESET_TENSORS or tensor_schema_changed) and TENSOR_DIR.exists():
    shutil.rmtree(TENSOR_DIR)
TENSOR_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESS_RUN_DIR.mkdir(parents=True, exist_ok=True)

%cd {ACESTEP_DIR}
!python train.py fixed \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --model-variant "{MODEL_VARIANT}" \
  --dataset-dir "{TENSOR_DIR}" \
  --output-dir "{PREPROCESS_RUN_DIR}" \
  --preprocess \
  --dual-stream \
  --dataset-json "{MANIFEST_PATH}" \
  --tensor-output "{TENSOR_DIR}" \
  --max-duration "{MAX_DURATION}" \
  --device "{DEVICE}" \
  --precision bf16

if _exit_code != 0:
    raise RuntimeError(f'dual-stream 전처리가 종료 코드 {_exit_code}로 실패했습니다.')
TENSOR_SCHEMA_PATH.write_text(TENSOR_SCHEMA + '\n', encoding='utf-8')

In [ ]:
import torch

tensor_files = sorted(TENSOR_DIR.glob('*.pt'))
assert tensor_files, '전처리 tensor가 생성되지 않았습니다.'
if SMOKE_TEST:
    assert len(tensor_files) == SMOKE_SONGS, (
        f'smoke tensor는 {SMOKE_SONGS}개여야 합니다: {len(tensor_files)}개'
    )
item = torch.load(tensor_files[0], map_location='cpu', weights_only=True)
required = [
    'motif_seed_latents', 'motif_seed_attention_mask',
    'motif_target_latents', 'motif_target_attention_mask',
    'vocal_target_latents', 'vocal_target_attention_mask',
    'motif_encoder_hidden_states', 'motif_encoder_attention_mask',
    'vocal_encoder_hidden_states', 'vocal_encoder_attention_mask',
]
for key in required:
    assert key in item, f'missing tensor: {key}'
    print(key, tuple(item[key].shape))

## 10. LoRA 학습

`SMOKE_TEST = True`이면 본학습과 분리된 3곡/1 epoch run에서 checkpoint 저장과 inference adapter 로딩까지 검증합니다. 통과 후 `False`로 바꾸면 500 epochs 본학습을 시작합니다. 곡 단위 80/20 train-validation 고정 분할을 사용합니다.

In [ ]:
import json
import os
import re
import torch

LOCAL_TENSOR_DIR = ACESTEP_DIR / 'mohim_dual_stream_tensors'
LOCAL_RUN_DIR = ACESTEP_DIR / f'mohim_lora_run_{LORA_VERSION}'
LEGACY_LOCAL_RUN_DIR = ACESTEP_DIR / 'mohim_lora_run'
EPOCHS = 1 if SMOKE_TEST else 500
SAVE_EVERY = 1 if SMOKE_TEST else 10
NUM_WORKERS = 0 if SMOKE_TEST else 4
LOG_EVERY = 1 if SMOKE_TEST else 10

# Side-Step safe-root 검사에 맞춰 학습 데이터와 resume checkpoint는 ACE-Step 내부에 둡니다.
if LOCAL_TENSOR_DIR.exists():
    shutil.rmtree(LOCAL_TENSOR_DIR)
if SMOKE_TEST or RESET_LORA_RUN:
    if LOCAL_RUN_DIR.exists():
        shutil.rmtree(LOCAL_RUN_DIR)
    if LORA_OUTPUT_DIR.exists():
        shutil.rmtree(LORA_OUTPUT_DIR)
    if DRIVE_CHECKPOINT_DIR.exists():
        shutil.rmtree(DRIVE_CHECKPOINT_DIR)
shutil.copytree(TENSOR_DIR, LOCAL_TENSOR_DIR)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# 버전 도입 전에 루트에 백업한 checkpoint는 v1으로 복사해 보존합니다.
if LORA_VERSION == 'v1' and not RESET_LORA_RUN:
    for legacy_checkpoint in DRIVE_CHECKPOINT_ROOT.glob('epoch_*'):
        if (legacy_checkpoint / 'training_state.pt').is_file():
            shutil.copytree(legacy_checkpoint, DRIVE_CHECKPOINT_DIR / legacy_checkpoint.name, dirs_exist_ok=True)

# 이전 실행에서 로컬에만 저장된 완료 checkpoint도 현재 버전 Drive에 보존합니다.
for run_dir in (LEGACY_LOCAL_RUN_DIR, LOCAL_RUN_DIR):
    for local_checkpoint in (run_dir / 'checkpoints').glob('epoch_*'):
        if (local_checkpoint / 'training_state.pt').is_file():
            shutil.copytree(local_checkpoint, DRIVE_CHECKPOINT_DIR / local_checkpoint.name, dirs_exist_ok=True)

def checkpoint_epoch(path):
    try:
        state = torch.load(path / 'training_state.pt', map_location='cpu', weights_only=False)
        return int(state.get('epoch', -1))
    except Exception:
        return -1

def valid_resume_checkpoint(path):
    required = (
        'training_state.pt', 'adapter_model.safetensors',
        'dual_stream_conditioner.pt', 'run_config.json', 'rng_state.pt',
    )
    return path.is_dir() and all((path / name).is_file() for name in required) and checkpoint_epoch(path) >= 0

resume_candidates = [
    DRIVE_CHECKPOINT_DIR / 'resume_latest',
    DRIVE_CHECKPOINT_DIR / 'resume_latest.previous',
    *DRIVE_CHECKPOINT_DIR.glob('epoch_*'),
]
drive_checkpoints = [path for path in resume_candidates if valid_resume_checkpoint(path)]
latest_checkpoint = max(drive_checkpoints, key=checkpoint_epoch, default=None)
resume_arg = ''
if latest_checkpoint is not None:
    local_resume_dir = LOCAL_RUN_DIR / 'resume_checkpoint'
    if local_resume_dir.exists():
        shutil.rmtree(local_resume_dir)
    shutil.copytree(latest_checkpoint, local_resume_dir)
    resume_arg = f'--resume-from {local_resume_dir}'
    print(f'resuming {LORA_VERSION} from:', latest_checkpoint)
else:
    print(f'no {LORA_VERSION} checkpoint found; starting from epoch 1')

# 패치된 trainer가 각 완료 epoch를 즉시 이 Drive 경로로 복사합니다.
os.environ['MOHIM_CHECKPOINT_BACKUP_DIR'] = str(DRIVE_CHECKPOINT_DIR)

%cd {ACESTEP_DIR}
!python train.py --plain --yes fixed \
  --checkpoint-dir "{CHECKPOINT_DIR}" \
  --model-variant "{MODEL_VARIANT}" \
  --dataset-dir "{LOCAL_TENSOR_DIR}" \
  --output-dir "{LOCAL_RUN_DIR}" \
  {resume_arg} \
  --dual-stream \
  --attention-type both \
  --rank 64 \
  --alpha 128 \
  --dropout 0.1 \
  --batch-size 1 \
  --gradient-accumulation 1 \
  --epochs "{EPOCHS}" \
  --save-every "{SAVE_EVERY}" \
  --lr 0.0003 \
  --val-split 0.2 \
  --shift 1.0 \
  --num-inference-steps 50 \
  --optimizer-type adamw8bit \
  --scheduler-type cosine_restarts \
  --warmup-steps 100 \
  --weight-decay 0.01 \
  --max-grad-norm 1.0 \
  --seed 42 \
  --num-workers "{NUM_WORKERS}" \
  --log-every "{LOG_EVERY}" \
  --device "{DEVICE}" \
  --precision bf16

if _exit_code != 0:
    raise RuntimeError(f'LoRA 학습 명령이 종료 코드 {_exit_code}로 실패했습니다.')
FINAL_ADAPTER_DIR = LOCAL_RUN_DIR / 'final'
assert (FINAL_ADAPTER_DIR / 'adapter_model.safetensors').is_file(), 'LoRA 학습 결과가 없습니다.'

if SMOKE_TEST:
    from peft import PeftModel
    from acestep.training_v2.dual_stream import DualStreamConditioner
    from acestep.training_v2.model_loader import load_decoder_for_training

    smoke_checkpoint = LOCAL_RUN_DIR / 'resume_latest'
    smoke_required = (
        'training_state.pt', 'adapter_model.safetensors', 'adapter_config.json',
        'dual_stream_config.json', 'dual_stream_conditioner.pt',
        'run_config.json', 'rng_state.pt',
    )
    smoke_missing = [name for name in smoke_required if not (smoke_checkpoint / name).is_file()]
    assert not smoke_missing, f'smoke checkpoint 파일이 없습니다: {smoke_missing}'

    smoke_model = load_decoder_for_training(
        CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16'
    )
    smoke_model.decoder = PeftModel.from_pretrained(
        smoke_model.decoder, str(FINAL_ADAPTER_DIR), is_trainable=False
    )
    smoke_conditioner_config = json.loads(
        (FINAL_ADAPTER_DIR / 'dual_stream_config.json').read_text(encoding='utf-8')
    )
    smoke_conditioner_config.pop('schema_version', None)
    smoke_conditioner = DualStreamConditioner(**smoke_conditioner_config)
    smoke_conditioner.load_state_dict(torch.load(
        FINAL_ADAPTER_DIR / 'dual_stream_conditioner.pt',
        map_location='cpu', weights_only=True,
    ))
    del smoke_conditioner, smoke_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('SMOKE TEST PASSED: training, checkpoint, and inference adapter loading')

shutil.copytree(LOCAL_RUN_DIR, LORA_OUTPUT_DIR, dirs_exist_ok=True)
print('saved to Drive:', LORA_OUTPUT_DIR)

In [ ]:
checkpoints = sorted(LORA_OUTPUT_DIR.rglob('*'))
print('output files:', len(checkpoints))
for path in checkpoints[-30:]:
    if path.is_file():
        print(path.relative_to(LORA_OUTPUT_DIR), path.stat().st_size)

## 11. 학습된 dual-stream LoRA inference

현재 `LORA_VERSION`에서 validation loss가 가장 낮은 checkpoint를 선택합니다. `/content/motif.wav`, 악기 이름, 언어, 가사와 생성 길이를 직접 입력해 ostinato/vocal stem을 공동 생성하며 `CFG_SCALE`로 두 prompt를 강화합니다.

In [ ]:
import gc
import json
import re
import soundfile as sf
import torch
from peft import PeftModel

from acestep.training.dataset_builder_modules.preprocess_encoder import run_encoder
from acestep.training.dataset_builder_modules.preprocess_lyrics import encode_lyrics
from acestep.training.dataset_builder_modules.preprocess_text import encode_text
from acestep.training_v2.dual_stream import DualStreamConditioner
from acestep.training_v2.dual_stream_inference import sample_dual_stream_latents
from acestep.training_v2.dual_stream_preprocess import encode_stem_latents
from acestep.training_v2.model_loader import (
    load_decoder_for_training, load_text_encoder, load_vae, unload_models,
)
from acestep.training_v2.preprocess_prompt import build_simple_prompt

CUSTOM_MOTIF_PATH = Path('/content/motif.wav')
CUSTOM_MOTIF_STEM = 'piano'
CUSTOM_LANGUAGE = 'English'
CUSTOM_LYRICS = '''
Paste the complete lyrics here.
'''
CUSTOM_OUTPUT_NAME = 'custom_song'
INFERENCE_DURATION_SECONDS = None  # None이면 motif.wav 길이
INFERENCE_STEPS = 50
INFERENCE_SEED = 42
CFG_SCALE = 7.0
checkpoint_pattern = re.compile(r'^epoch_(\d+)_loss_([0-9.]+)$')
checkpoint_candidates = []
checkpoint_search_dirs = [DRIVE_CHECKPOINT_DIR]
if LORA_VERSION == 'v1':
    checkpoint_search_dirs.append(DRIVE_CHECKPOINT_ROOT)
for checkpoint_dir in checkpoint_search_dirs:
    for checkpoint_path in checkpoint_dir.glob('epoch_*_loss_*'):
        match = checkpoint_pattern.match(checkpoint_path.name)
        if (
            match
            and (checkpoint_path / 'adapter_model.safetensors').is_file()
            and (checkpoint_path / 'dual_stream_conditioner.pt').is_file()
        ):
            checkpoint_candidates.append((float(match.group(2)), int(match.group(1)), checkpoint_path))
    for checkpoint_path in (checkpoint_dir / 'resume_latest', checkpoint_dir / 'resume_latest.previous'):
        config_path = checkpoint_path / 'run_config.json'
        if (
            (checkpoint_path / 'adapter_model.safetensors').is_file()
            and (checkpoint_path / 'dual_stream_conditioner.pt').is_file()
            and config_path.is_file()
        ):
            run_config = json.loads(config_path.read_text(encoding='utf-8'))
            validation_loss = run_config.get('val_loss')
            epoch = checkpoint_epoch(checkpoint_path)
            if validation_loss is not None and epoch >= 0:
                checkpoint_candidates.append((float(validation_loss), epoch, checkpoint_path))
assert checkpoint_candidates, f'{LORA_VERSION} checkpoint가 없습니다: {DRIVE_CHECKPOINT_DIR}'
BEST_LOSS, BEST_EPOCH, BEST_ADAPTER_DIR = min(checkpoint_candidates, key=lambda item: item[0])

assert CUSTOM_MOTIF_PATH.is_file(), f'motif 파일이 없습니다: {CUSTOM_MOTIF_PATH}'
assert CUSTOM_MOTIF_STEM.strip(), 'CUSTOM_MOTIF_STEM을 입력하세요.'
assert CUSTOM_LYRICS.strip(), 'CUSTOM_LYRICS를 입력하세요.'
assert CFG_SCALE >= 1.0, 'CFG_SCALE은 1 이상이어야 합니다.'

dtype = torch.bfloat16
motif_stem = CUSTOM_MOTIF_STEM.strip().lower()
motif_duration = sf.info(str(CUSTOM_MOTIF_PATH)).duration
generation_duration = INFERENCE_DURATION_SECONDS or motif_duration
ostinato_caption = (
    f'Isolated {motif_stem} stem performing a recurring ostinato in a pop style. '
    f'{motif_stem.capitalize()} only; no vocals and no other instruments.'
)
vocal_caption = (
    f'Isolated vocal stem for an {CUSTOM_LANGUAGE.strip()} pop song, singing the provided lyrics. '
    'Vocals only; no instrumental accompaniment and no musical instruments.'
)

print(f'version: {LORA_VERSION}, best epoch: {BEST_EPOCH}, validation loss: {BEST_LOSS:.4f}')
print('adapter:', BEST_ADAPTER_DIR)
print('custom motif:', CUSTOM_MOTIF_PATH)
print('ostinato caption:', ostinato_caption)
print('vocal caption:', vocal_caption)
print('lyrics characters:', len(CUSTOM_LYRICS.strip()))
print('generation duration:', generation_duration, 'seconds')
print('CFG scale:', CFG_SCALE)

vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
custom_motif_latents = encode_stem_latents(
    str(CUSTOM_MOTIF_PATH), vae, dtype, max_duration=motif_duration
)
latent_fps = custom_motif_latents.shape[0] / motif_duration
output_frames = max(1, round(generation_duration * latent_fps))
custom_motif_mask = torch.ones(
    custom_motif_latents.shape[0], dtype=dtype
)
unload_models(vae)
del vae

text_tokenizer, text_encoder = load_text_encoder(
    CHECKPOINT_DIR, device=DEVICE, precision='bf16'
)
def encode_manual_prompt(caption, lyrics):
    prompt = build_simple_prompt(
        {
            'caption': caption, 'duration': generation_duration,
            'bpm': None, 'timesignature': '', 'keyscale': '',
        }
    )
    text_hs, text_mask = encode_text(
        text_encoder, text_tokenizer, prompt, DEVICE, dtype
    )
    lyric_hs, lyric_mask = encode_lyrics(
        text_encoder, text_tokenizer, lyrics, DEVICE, dtype
    )
    return text_hs, text_mask, lyric_hs, lyric_mask

motif_text = encode_manual_prompt(ostinato_caption, '[Instrumental]')
vocal_text = encode_manual_prompt(vocal_caption, CUSTOM_LYRICS.strip())
unload_models(text_encoder)
del text_encoder, text_tokenizer

model = load_decoder_for_training(
    CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16'
)
motif_encoder_hs, motif_encoder_mask = run_encoder(
    model, motif_text[0], motif_text[1], motif_text[2], motif_text[3], DEVICE, dtype
)
vocal_encoder_hs, vocal_encoder_mask = run_encoder(
    model, vocal_text[0], vocal_text[1], vocal_text[2], vocal_text[3], DEVICE, dtype
)
del motif_text, vocal_text
model.decoder = PeftModel.from_pretrained(
    model.decoder, str(BEST_ADAPTER_DIR), is_trainable=False
).to(device=DEVICE, dtype=dtype).eval()

conditioner_config = json.loads(
    (BEST_ADAPTER_DIR / 'dual_stream_config.json').read_text(encoding='utf-8')
)
conditioner_config.pop('schema_version', None)
conditioner = DualStreamConditioner(**conditioner_config).to(DEVICE, dtype=dtype).eval()
conditioner.load_state_dict(
    torch.load(
        BEST_ADAPTER_DIR / 'dual_stream_conditioner.pt',
        map_location=DEVICE,
        weights_only=True,
    )
)

with torch.inference_mode():
    generated_motif_latents, generated_vocal_latents = sample_dual_stream_latents(
        decoder=model.decoder,
        conditioner=conditioner,
        motif_encoder_hidden_states=motif_encoder_hs.to(DEVICE, dtype=dtype),
        motif_encoder_attention_mask=motif_encoder_mask.to(DEVICE, dtype=dtype),
        vocal_encoder_hidden_states=vocal_encoder_hs.to(DEVICE, dtype=dtype),
        vocal_encoder_attention_mask=vocal_encoder_mask.to(DEVICE, dtype=dtype),
        motif_seed_latents=custom_motif_latents.unsqueeze(0).to(DEVICE, dtype=dtype),
        motif_seed_attention_mask=custom_motif_mask.unsqueeze(0).to(DEVICE),
        null_condition_emb=model.null_condition_emb.to(DEVICE, dtype=dtype),
        output_frames=output_frames,
        latent_dim=custom_motif_latents.shape[-1],
        steps=INFERENCE_STEPS,
        seed=INFERENCE_SEED,
        guidance_scale=CFG_SCALE,
    )
generated_motif_latents = generated_motif_latents.cpu()
generated_vocal_latents = generated_vocal_latents.cpu()
del model, conditioner, custom_motif_latents
gc.collect()
torch.cuda.empty_cache()
print('custom motif-conditioned latent generation complete')

In [ ]:
import math
import torch.nn.functional as F
import soundfile as sf
from IPython.display import Audio, display
from acestep.training_v2.model_loader import load_vae

def decode_latents_tiled(vae, latents_btc, chunk_frames=256, overlap=64):
    latents = latents_btc.transpose(1, 2)
    latent_frames = latents.shape[-1]
    stride = chunk_frames - 2 * overlap
    if stride <= 0:
        raise ValueError('chunk_frames must be larger than twice overlap')
    decoded = []
    steps = math.ceil(latent_frames / stride)
    upsample_factor = None
    for index in range(steps):
        core_start = index * stride
        core_end = min(core_start + stride, latent_frames)
        window_start = max(0, core_start - overlap)
        window_end = min(latent_frames, core_end + overlap)
        chunk = latents[:, :, window_start:window_end].to(DEVICE, dtype=vae.dtype)
        with torch.inference_mode():
            audio = vae.decode(chunk).sample
        if upsample_factor is None:
            upsample_factor = audio.shape[-1] / chunk.shape[-1]
        trim_start = round((core_start - window_start) * upsample_factor)
        trim_end = round((window_end - core_end) * upsample_factor)
        end = audio.shape[-1] - trim_end if trim_end else audio.shape[-1]
        decoded.append(audio[:, :, trim_start:end].float().cpu())
        del chunk, audio
    return torch.cat(decoded, dim=-1)

vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
motif_audio = decode_latents_tiled(vae, generated_motif_latents)
vocal_audio = decode_latents_tiled(vae, generated_vocal_latents)
del vae, generated_motif_latents, generated_vocal_latents
gc.collect()
torch.cuda.empty_cache()

output_sample_rate = 48000
target_samples = round(generation_duration * output_sample_rate)
def match_length(audio):
    if audio.shape[-1] < target_samples:
        audio = F.pad(audio, (0, target_samples - audio.shape[-1]))
    return audio[:, :, :target_samples]
motif_audio = match_length(motif_audio)
vocal_audio = match_length(vocal_audio)
preview_mix = motif_audio + vocal_audio
preview_mix = preview_mix / preview_mix.abs().amax().clamp_min(1.0)

track_output_dir = INFERENCE_OUTPUT_DIR / BEST_ADAPTER_DIR.name / CUSTOM_OUTPUT_NAME
track_output_dir.mkdir(parents=True, exist_ok=True)
outputs = {
    'motif': track_output_dir / 'generated_motif.wav',
    'vocals': track_output_dir / 'generated_vocals.wav',
    'preview_mix': track_output_dir / 'generated_preview_mix.wav',
}
for name, path in outputs.items():
    audio = {'motif': motif_audio, 'vocals': vocal_audio, 'preview_mix': preview_mix}[name]
    sf.write(path, audio.squeeze(0).transpose(0, 1).numpy(), output_sample_rate)
    print(name, path, sf.info(path).duration, 'seconds')
    display(Audio(filename=str(path)))